In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [2]:



# ── Palette ──────────────────────────────────────────────────────────────────
BG      = "#0D1117"
CARD    = "#161B22"
ACCENT  = "#00D4FF"
WARM    = "#FF6B6B"
GREEN   = "#3DDC97"
PURPLE  = "#C084FC"
TEXT    = "#E6EDF3"
MUTED   = "#8B949E"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.edgecolor": "#30363D", "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": "#21262D",
    "grid.linewidth": 0.6, "font.family": "DejaVu Sans",
})

# ── Load & encode ─────────────────────────────────────────────────────────────


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = model.score(X_test, y_test)

print(f"MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Scatter Plots  (BMI, Age, Smoker vs Charges)
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(BG)
fig.suptitle("How Key Factors Drive Insurance Charges",
             fontsize=18, fontweight="bold", color=TEXT, y=1.02)

# 1. BMI vs Charges
ax = axes[0]
colors = [ACCENT if s == "yes" else WARM for s in df["smoker"]]
ax.scatter(df["bmi"], df["charges"], c=colors, alpha=0.55, s=30, edgecolors="none")
m, b = np.polyfit(df["bmi"], df["charges"], 1)
x_line = np.linspace(df["bmi"].min(), df["bmi"].max(), 200)
ax.plot(x_line, m*x_line + b, color=GREEN, lw=2, label="Trend")
ax.set_title("BMI vs Charges", color=TEXT, fontsize=13, pad=10)
ax.set_xlabel("BMI", fontsize=11)
ax.set_ylabel("Charges ($)", fontsize=11)
ax.grid(True, alpha=0.3)
patch1 = mpatches.Patch(color=ACCENT, label="Smoker")
patch2 = mpatches.Patch(color=WARM,  label="Non-smoker")
ax.legend(handles=[patch1, patch2], facecolor=CARD, edgecolor="#30363D",
          labelcolor=TEXT, fontsize=9)

# 2. Age vs Charges
ax = axes[1]
ax.scatter(df["age"], df["charges"], c=colors, alpha=0.55, s=30, edgecolors="none")
m2, b2 = np.polyfit(df["age"], df["charges"], 1)
x_line2 = np.linspace(df["age"].min(), df["age"].max(), 200)
ax.plot(x_line2, m2*x_line2 + b2, color=GREEN, lw=2)
ax.set_title("Age vs Charges", color=TEXT, fontsize=13, pad=10)
ax.set_xlabel("Age", fontsize=11)
ax.set_ylabel("Charges ($)", fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(handles=[patch1, patch2], facecolor=CARD, edgecolor="#30363D",
          labelcolor=TEXT, fontsize=9)

# 3. Smoking Status vs Charges  (violin)
ax = axes[2]
smoker_yes = df[df["smoker"] == "yes"]["charges"]
smoker_no  = df[df["smoker"] == "no"]["charges"]
parts = ax.violinplot([smoker_no, smoker_yes], positions=[0, 1],
                      showmedians=True, showextrema=True)
for i, (pc, col) in enumerate(zip(parts["bodies"], [WARM, ACCENT])):
    pc.set_facecolor(col); pc.set_alpha(0.7)
parts["cmedians"].set_color(GREEN); parts["cmedians"].set_linewidth(2)
parts["cmins"].set_color(MUTED);   parts["cmaxes"].set_color(MUTED)
parts["cbars"].set_color(MUTED)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Non-Smoker", "Smoker"], color=TEXT, fontsize=11)
ax.set_title("Smoking Status vs Charges", color=TEXT, fontsize=13, pad=10)
ax.set_ylabel("Charges ($)", fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig1_scatter_plots.png",
            dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("fig1 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Correlation Heatmap
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(BG)
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt=".2f", linewidths=0.5,
            linecolor="#0D1117", ax=ax,
            annot_kws={"size": 10, "color": TEXT},
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=15,
             fontweight="bold", color=TEXT, pad=14)
ax.tick_params(colors=TEXT, labelsize=10)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig2_correlation_heatmap.png",
            dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("fig2 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Actual vs Predicted  +  Residuals
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG)

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred, color=ACCENT, alpha=0.5, s=28, edgecolors="none")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color=GREEN, lw=2, ls="--", label="Perfect fit")
ax.set_title("Actual vs Predicted Charges", color=TEXT, fontsize=13, pad=10)
ax.set_xlabel("Actual Charges ($)", fontsize=11)
ax.set_ylabel("Predicted Charges ($)", fontsize=11)
ax.legend(facecolor=CARD, edgecolor="#30363D", labelcolor=TEXT)
ax.grid(True, alpha=0.3)

# Residual plot
ax = axes[1]
residuals = y_test - y_pred
ax.scatter(y_pred, residuals, color=PURPLE, alpha=0.5, s=28, edgecolors="none")
ax.axhline(0, color=GREEN, lw=2, ls="--")
ax.set_title("Residual Plot", color=TEXT, fontsize=13, pad=10)
ax.set_xlabel("Predicted Charges ($)", fontsize=11)
ax.set_ylabel("Residuals ($)", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig3_model_evaluation.png",
            dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("fig3 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Feature Importance (coefficients)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)

coef_df = pd.DataFrame({"Feature": X.columns,
                         "Coefficient": model.coef_}).sort_values("Coefficient")
bar_colors = [WARM if c < 0 else ACCENT for c in coef_df["Coefficient"]]
bars = ax.barh(coef_df["Feature"], coef_df["Coefficient"],
               color=bar_colors, edgecolor="none", height=0.55)

for bar, val in zip(bars, coef_df["Coefficient"]):
    ax.text(val + (200 if val >= 0 else -200), bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center",
            ha="left" if val >= 0 else "right",
            color=TEXT, fontsize=9)

ax.axvline(0, color=MUTED, lw=1, ls="--")
ax.set_title("Linear Regression — Feature Coefficients",
             color=TEXT, fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Coefficient Value ($)", fontsize=11)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig4_feature_coefficients.png",
            dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("fig4 saved")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Metrics Summary Card
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.axis("off")

metrics = [
    ("MAE",  f"${mae:,.2f}",  ACCENT,  "Mean Absolute Error\n(avg prediction error)"),
    ("RMSE", f"${rmse:,.2f}", WARM,    "Root Mean Squared Error\n(penalises large errors)"),
    ("R²",   f"{r2:.4f}",    GREEN,   "Coefficient of Determination\n(variance explained)"),
]

for i, (label, value, color, desc) in enumerate(metrics):
    x = 0.15 + i * 0.33
    rect = mpatches.FancyBboxPatch((x - 0.12, 0.1), 0.24, 0.8,
                                    boxstyle="round,pad=0.03",
                                    linewidth=2, edgecolor=color,
                                    facecolor=CARD, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x, 0.72, label,  ha="center", va="center", fontsize=18,
            fontweight="bold", color=color,  transform=ax.transAxes)
    ax.text(x, 0.50, value,  ha="center", va="center", fontsize=22,
            fontweight="bold", color=TEXT,   transform=ax.transAxes)
    ax.text(x, 0.24, desc,   ha="center", va="center", fontsize=9,
            color=MUTED, transform=ax.transAxes, linespacing=1.4)

ax.set_title("Model Performance Metrics  —  Linear Regression",
             fontsize=14, fontweight="bold", color=TEXT, pad=16)
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig5_metrics_card.png",
            dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("fig5 saved")
print(f"\nDONE  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}")

NameError: name 'X' is not defined

In [ ]:
df = pd.read_csv("/mnt/user-data/uploads/insurance.csv")
df.columns = df.columns.str.strip()


In [ ]:
le = LabelEncoder()
df_enc = df.copy()
for col in ["sex", "smoker", "region"]:
    df_enc[col] = le.fit_transform(df_enc[col])

X = df_enc.drop("charges", axis=1)
y = df_enc["charges"]
